<a href="https://colab.research.google.com/github/PrantoMondol11/2.1/blob/main/Knowledage%20Distilation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [3]:
transform=transforms.ToTensor()
train_data=datasets.MNIST(root='./data',train=True,download=True,transform=transform)
train_loader=DataLoader(train_data,batch_size=64,shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 45.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.15MB/s]


In [4]:
teacher=nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28,256),
    nn.ReLU(),
    nn.Linear(256,10)
)

In [5]:
student=nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28,64),
    nn.ReLU(),
    nn.Linear(64,10)
)

In [6]:
optimizer_t=optim.Adam(teacher.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()

for epoch in range(2):
  for x,y in train_loader:
    pred=teacher(x)
    loss=loss_fn(pred,y)

    optimizer_t.zero_grad()
    loss.backward()
    optimizer_t.step()

In [9]:

optimizer=optim.Adam(student.parameters(),lr=0.001)
temperature=3
alpha=.5
ce_loss=nn.CrossEntropyLoss()
kl_loss=nn.KLDivLoss(reduce='batchmean')

In [10]:
for epoch in range(2):
    for x, y in train_loader:
       with torch.no_grad():
            teacher_logits = teacher(x)
       student_logits = student(x)
       loss_hard = ce_loss(student_logits, y)
       soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
       soft_student = F.log_softmax(student_logits / temperature, dim=1)
       loss_soft = kl_loss(soft_student, soft_teacher)
       loss = alpha * loss_hard + (1 - alpha) * loss_soft
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:558: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  return F.kl_div(


In [11]:
correct = 0
total = 0

with torch.no_grad():
    for x, y in train_loader:
        outputs = student(x)
        _, predicted = torch.max(outputs, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

print("Accuracy:", correct / total)

Accuracy: 0.9551666666666667


In [12]:
from sklearn.metrics import accuracy_score
y_true = []
y_pred = []

with torch.no_grad():
    for x, y in train_loader:
        outputs = student(x)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(y.numpy())
        y_pred.extend(predicted.numpy())
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)

Accuracy: 0.9551666666666667
